# tJ Julia Function Tests

Simple sanity tests for the first Julia port in `tJ.jl`. These cells are intentionally unexecuted.

In [1]:
using Test
using SparseArrays

module_path = isfile("tJ.jl") ? "tJ.jl" : joinpath("tJ_julia", "tJ.jl")
include(module_path)
using .TJModels

## Lattice Construction

Check the honeycomb supercell bookkeeping and nearest-neighbor list shape.

In [2]:
@testset "Lattice construction" begin
    lat = Lattice(2, 0, 0, 2)

    @test lat.M == 8
    @test length(lat.Asites) == div(lat.M, 2)
    @test length(lat.Bsites) == div(lat.M, 2)
    @test length(lat.lattice_sites) == lat.M
    @test length(lat.lattice_hash) == lat.M
    @test length(lat.lattice_hash_inv) == lat.M

    @test all(haskey(lat.lattice_hash, (round(Int, site[1]), round(Int, site[2]))) for site in lat.lattice_sites)

    NNs = kneighbours(lat, 1)
    @test length(NNs) == div(3 * lat.M, 2)
    @test length(unique(NNs)) == length(NNs)
    @test all(i < j for (i, j) in NNs)

    NN2s = kneighbours(lat, 2)
    @test !isempty(NN2s)
    @test length(unique(NN2s)) == length(NN2s)
    @test all(i < j for (i, j) in NN2s)
end

Test Summary:        | Pass  Total  Time
Lattice construction |   13     13  2.2s


Test.DefaultTestSet("Lattice construction", Any[], 13, false, false, true, 1.777950608940311e9, 1.777950611175786e9, false, "In[2]", Random.Xoshiro(0x253a65b5c10608e6, 0x6ee4856feca50ea4, 0x01baa094c001c76c, 0x9b000a202cd6486c, 0xb78d5fce80da14e2))

## Basis And Local Operators

Check model attributes, basis counts, fermion hopping signs, spin flips, and diagonal operators.

In [3]:
@testset "Basis and operators" begin
    model = TJ(3, 2, 2, 0, 0, 2)

    @test model.M == 8
    @test model.h == 3
    @test model.N == 5
    @test model.dim == binomial(model.M, model.nup) * binomial(model.M - model.nup, model.ndown)
    @test length(model.basis) == model.dim
    @test length(model.hashmap) == model.dim

    @test all(length(state) == model.M for state in model.basis)
    @test all(count(==(Int8(1)), state) == model.nup for state in model.basis)
    @test all(count(==(Int8(-1)), state) == model.ndown for state in model.basis)
    @test all(count(==(Int8(0)), state) == model.h for state in model.basis)

    state = Int8[1, 0, -1, 0, 1, -1, 0, 1]

    hopped, sign = cicjdagger(model, 1, 1, 2, state)
    @test hopped == Int8[0, 1, -1, 0, 1, -1, 0, 1]
    @test sign == 1

    blocked, blocked_sign = cicjdagger(model, 1, 1, 5, state)
    @test blocked === nothing
    @test blocked_sign == 0

    @test fermion_sign_hop(model, 1, 4, Int8[1, -1, 1, 0, 0, 0, 0, 0]) == 1
    @test fermion_sign_hop(model, 1, 3, Int8[1, -1, 0, 0, 0, 0, 0, 0]) == -1

    flipped = XiXj(model, 1, 3, state)
    @test flipped == Int8[-1, 0, 1, 0, 1, -1, 0, 1]
    @test XiXj(model, 1, 5, state) === nothing

    @test ZiZj(model, 1, 3, state) == -0.25
    @test ZiZj(model, 1, 5, state) == 0.25
    @test ninj(model, 1, 3, state) == 1
    @test ninj(model, 1, 2, state) == 0
    @test nh(model, 2, state) == 1
    @test nh(model, 1, state) == 0
end

Test Summary:       | Pass  Total  Time
Basis and operators |   24     24  0.8s


Test.DefaultTestSet("Basis and operators", Any[], 24, false, false, true, 1.777950614085862e9, 1.77795061488137e9, false, "In[3]", Random.Xoshiro(0x253a65b5c10608e6, 0x6ee4856feca50ea4, 0x01baa094c001c76c, 0x9b000a202cd6486c, 0xb78d5fce80da14e2))

## Correlation Matrices

Check that spin and hole correlation helpers return diagonal sparse matrices of the expected size.

In [4]:
@testset "Correlation matrices" begin
    model = TJ(3, 2, 2, 0, 0, 2)
    sample_basis = model.basis[1:10]

    Css = spin_corr(model, sample_basis)
    @test size(Css) == (10, 10)
    @test issparse(Css)
    @test all(Css[i, j] == 0 for i in axes(Css, 1), j in axes(Css, 2) if i != j)

    Chh1 = hole_corr(model, sample_basis, 1)
    @test size(Chh1) == (10, 10)
    @test issparse(Chh1)
    @test all(Chh1[i, j] == 0 for i in axes(Chh1, 1), j in axes(Chh1, 2) if i != j)
end

Test Summary:        | Pass  Total  Time
Correlation matrices |    6      6  0.5s


Test.DefaultTestSet("Correlation matrices", Any[], 6, false, false, true, 1.77795061584879e9, 1.777950616324804e9, false, "In[4]", Random.Xoshiro(0x253a65b5c10608e6, 0x6ee4856feca50ea4, 0x01baa094c001c76c, 0x9b000a202cd6486c, 0xb78d5fce80da14e2))

## Basis Reordering

Check that basis reordering preserves the basis size and rebuilds a complete hashmap.

In [5]:
@testset "Basis reordering" begin
    balanced = TJ(2, 2, 2, 0, 0, 2)
    original_dim = balanced.dim

    reorder_basis!(balanced)

    @test balanced.dim == original_dim
    @test length(balanced.basis) == original_dim
    @test length(balanced.hashmap) == original_dim
    @test all(get(balanced.hashmap, Tuple(state), 0) > 0 for state in balanced.basis)
end

Test Summary:    | Pass  Total  Time
Basis reordering |    4      4  0.2s


Test.DefaultTestSet("Basis reordering", Any[], 4, false, false, true, 1.777950617357155e9, 1.777950617557191e9, false, "In[5]", Random.Xoshiro(0x253a65b5c10608e6, 0x6ee4856feca50ea4, 0x01baa094c001c76c, 0x9b000a202cd6486c, 0xb78d5fce80da14e2))

## Hamiltonian Triplets And Assembly

Check the Python-compatible `Hamiltonian` return shape, sparse assembly, and the more general degree-based interface.

In [6]:
@testset "Hamiltonian construction" begin
    small = TJ(1, 1, 2, 0, 0, 2)

    Hparams = Hamiltonian(small)
    @test length(Hparams) == 12
    @test all(x isa Vector for x in Hparams)

    Ht, Ht2, HJ, HJ2 = assembleH(small, Hparams)
    @test size(Ht) == (small.dim, small.dim)
    @test size(Ht2) == (small.dim, small.dim)
    @test size(HJ) == (small.dim, small.dim)
    @test size(HJ2) == (small.dim, small.dim)
    @test issparse(Ht)
    @test issparse(Ht2)
    @test issparse(HJ)
    @test issparse(HJ2)

    @test nnz(Ht - transpose(Ht)) == 0
    @test nnz(Ht2 - transpose(Ht2)) == 0
    @test nnz(HJ - transpose(HJ)) == 0
    @test nnz(HJ2 - transpose(HJ2)) == 0

    terms = hamiltonian_by_degree(small; degrees=(1, 2, 3))
    @test Set(keys(terms.t)) == Set([1, 2, 3])
    @test Set(keys(terms.J)) == Set([1, 2, 3])

    mats = assemble_by_degree(small, terms)
    @test all(size(mats.t[d]) == (small.dim, small.dim) for d in (1, 2, 3))
    @test all(size(mats.J[d]) == (small.dim, small.dim) for d in (1, 2, 3))
end

Test Summary:            | Pass  Total  Time
Hamiltonian construction |   18     18  1.2s


Test.DefaultTestSet("Hamiltonian construction", Any[], 18, false, false, true, 1.777950618541052e9, 1.777950619692762e9, false, "In[6]", Random.Xoshiro(0x253a65b5c10608e6, 0x6ee4856feca50ea4, 0x01baa094c001c76c, 0x9b000a202cd6486c, 0xb78d5fce80da14e2))

## 8-Site tJ Hamiltonian

Assemble the nearest-neighbor hopping and nearest-neighbor J pieces separately for an 8-site system with 3 spin-up electrons, 3 spin-down electrons, and 2 holes. No next-nearest-neighbor J term is included.

In [2]:
model8 = TJ(3, 3, 2, 0, 0, 2)

# First-neighbor terms only: Ht_nn is the hopping operator and HJ_nn is the J operator.
terms8 = hamiltonian_by_degree(model8; degrees=(1,))
mats8 = assemble_by_degree(model8, terms8)

Ht_nn = mats8.t[1]
HJ_nn = mats8.J[1]

# Use arbitrary coefficients. For the conventional t-J sign convention,
# call hamiltonian8(-t_value, J_value).
hamiltonian8(tcoef, Jcoef) = tcoef * Ht_nn + Jcoef * HJ_nn

t_value = 1.0
J_value = 0.15
H8 = hamiltonian8(-t_value, J_value)

(model8=model8, Ht_nn=Ht_nn, HJ_nn=HJ_nn, H8=H8)

(model8 = TJ(Lattice((3.0, 0.0), (0.0, 3.0), 3.0, (3.0, 0.0), (1.5, 2.598076211353316), (1.5, 0.8660254037844386), (0.0, -1.7320508075688772), (-1.5, 0.8660254037844386), (6.0, 0.0), (3.0, 5.196152422706632), 1.0471975511965979, (6.0, 0.0), (0.0, 6.0), 8, 2, 0, 0, 2, [1.0 0.5; 0.0 0.8660254037844386], [1.0 -0.5773502691896258; 0.0 1.1547005383792517], [6.0 0.0; 0.0 6.0], [0.16666666666666666 0.0; 0.0 0.16666666666666666], 0, 6, 0, 6, [(0.0, 0.0), (3.0, 0.0), (0.0, 3.0), (3.0, 3.0)], [(1.0, 1.0), (4.0, 1.0), (1.0, 4.0), (4.0, 4.0)], [(0.0, 0.0), (1.0, 1.0), (3.0, 0.0), (4.0, 1.0), (0.0, 3.0), (1.0, 4.0), (3.0, 3.0), (4.0, 4.0)], Dict((0, 0) => 1, (1, 1) => 2, (3, 3) => 7, (4, 1) => 4, (0, 3) => 5, (1, 4) => 6, (3, 0) => 3, (4, 4) => 8), Dict(5 => (0, 3), 4 => (4, 1), 6 => (1, 4), 7 => (3, 3), 2 => (1, 1), 8 => (4, 4), 3 => (3, 0), 1 => (0, 0))), 3, 3, 2, "honeycomb", true, 0.0, 6, -1, 560, Vector{Int8}[[-1, -1, -1, 0, 0, 1, 1, 1], [-1, -1, -1, 0, 1, 0, 1, 1], [-1, -1, -1, 0, 1, 1, 0, 1]

## Diagonalization

Use sparse iterative diagonalization for sparse Hamiltonians, returning the smallest `k` eigenvalues and eigenvectors. For dense Hamiltonians, use full diagonalization.

In [3]:
try
    import Arpack
catch err
    @warn "Arpack.jl is not available; sparse diagonalization will fail until it is installed." exception=(err, catch_backtrace())
end

function diagonalize_hamiltonian(H; k=6, kwargs...)
    if issparse(H)
        @isdefined(Arpack) || throw(ArgumentError("Sparse diagonalization requires Arpack.jl. Install it with `import Pkg; Pkg.add(\"Arpack\")`."))

        nev = min(k, size(H, 1) - 2)
        nev > 0 || throw(ArgumentError("Sparse diagonalization needs matrix size at least 3."))

        evals, evecs, nconv, niter, nmult, resid = Arpack.eigs(
            H;
            nev=nev,
            which=:SR,
            ritzvec=true,
            kwargs...,
        )

        order = sortperm(real.(evals))
        return (
            eigenvalues=evals[order],
            eigenvectors=evecs[:, order],
            nconv=nconv,
            niter=niter,
            nmult=nmult,
            residual=resid,
        )
    else
        A = Matrix(H)
        F = ishermitian(A) ? eigen(Hermitian(A)) : eigen(A)
        return (
            eigenvalues=F.values,
            eigenvectors=F.vectors,
        )
    end
end

diagonalize_hamiltonian (generic function with 1 method)

In [4]:
k_eigs = 6
spectrum8 = diagonalize_hamiltonian(H8; k=k_eigs)

lowest_eigenvalues8 = spectrum8.eigenvalues
lowest_eigenvectors8 = spectrum8.eigenvectors

(lowest_eigenvalues8=lowest_eigenvalues8, lowest_eigenvectors8=lowest_eigenvectors8)

(lowest_eigenvalues8 = [-5.049420618984206, -4.999734506703198, -4.999734506703188, -4.90450794730692, -4.904507947306904, -4.903363503106195], lowest_eigenvectors8 = [0.08771062816908402 -0.008484889962077372 … 0.0969793349895119 5.62682333835068e-15; 0.07547406574463526 0.006974209254010189 … 0.08920918564249267 0.004947850704172866; … ; -0.07547406574463995 -0.006974209253999867 … 0.08920918564248943 0.004947850704172615; -0.08771062816908938 0.008484889962089593 … 0.09697933498950537 5.2970839537788005e-15])

## Neural-Network Ground State

Train the Julia neural-network variational ansatz on the already assembled 8-site tJ Hamiltonian. This uses the same convention as above: `H8 = -t_value * Ht_nn + J_value * HJ_nn`.

In [5]:
using LinearAlgebra

tjnet_path = isfile("tJNetRC.jl") ? "tJNetRC.jl" : joinpath("tJ_julia", "tJNetRC.jl")
include(tjnet_path)
using .TJNetRC

In [6]:
TJNetRC.gpu_available()

true

In [14]:
nn_params8 = Dict{String, Any}(
    "n_neurons" => 128,
    "dropout" => 0.0,
    "epochs" => 500,
    "learning_rate" => 1.0,
    "history_size" => 200,
    "model_path" => "model8_tJ_nn.jls",
    "load_from" => nothing,
    "loss_diff" => 1.0e-4,
    "stagnation" => 20,
    "print_every" => 5,
    "verbose" => true,
    "exploit_symmetry" => false,
)

nn_params8["device"] = "gpu"      # or "auto" / "cpu"
nn_params8["nn_optimizer"] = "adam"
# params["nn_learning_rate"] = 1e-3

"adam"

In [16]:
training_tcoef8 = -t_value
training_Jcoef8 = J_value

# This is the actual training call. It is intentionally left unrun here.
psi0_nn8, E0_nn8 = TJNetRC.trainNN(
    nn_params8,
    model8,
    training_tcoef8,
    training_Jcoef8,
    Ht_nn,
    HJ_nn;
    seed=12345,
)

exact_E0_8 = @isdefined(lowest_eigenvalues8) ? minimum(real.(lowest_eigenvalues8)) : missing
energy_error8 = exact_E0_8 === missing ? missing : E0_nn8 - exact_E0_8
overlap8 = @isdefined(lowest_eigenvectors8) ? maximum(abs.(lowest_eigenvectors8' * psi0_nn8)) : missing

(E0_nn8=E0_nn8, exact_E0_8=exact_E0_8, energy_error8=energy_error8, overlap8=overlap8, norm_psi0=norm(psi0_nn8))

Neural network training device: gpu
epoch 5 completed, maximum remaining epochs: 495	Current loss: -3.4861080646514893
epoch 10 completed, maximum remaining epochs: 490	Current loss: -3.606088399887085
epoch 15 completed, maximum remaining epochs: 485	Current loss: -3.7699790000915527
epoch 20 completed, maximum remaining epochs: 480	Current loss: -4.0023674964904785
epoch 25 completed, maximum remaining epochs: 475	Current loss: -4.197379112243652
epoch 30 completed, maximum remaining epochs: 470	Current loss: -4.380356311798096
epoch 35 completed, maximum remaining epochs: 465	Current loss: -4.507056713104248
epoch 40 completed, maximum remaining epochs: 460	Current loss: -4.5996222496032715
epoch 45 completed, maximum remaining epochs: 455	Current loss: -4.686657905578613
epoch 50 completed, maximum remaining epochs: 450	Current loss: -4.769839763641357
epoch 55 completed, maximum remaining epochs: 445	Current loss: -4.84656286239624
epoch 60 completed, maximum remaining epochs: 440

(E0_nn8 = -5.049359321594238, exact_E0_8 = -5.049420618984206, energy_error8 = 6.129738996740741e-5, overlap8 = 0.9994482265660839, norm_psi0 = 1.0f0)

## Monte Carlo Sampling For The 8-Site Hamiltonian

Check the Metropolis sampler, matrix-free local energies, MC energy estimates, and a tiny VMC training update on the same 8-site tJ Hamiltonian defined above.

In [11]:
using Random

@testset "Monte Carlo sampling for 8-site tJ" begin
    mc_tcoef8 = -t_value
    mc_Jcoef8 = J_value
    H8_mc = mc_tcoef8 .* Ht_nn .+ mc_Jcoef8 .* HJ_nn
    @test nnz(H8_mc - H8) == 0

    uniform_net8(x) = ones(Float32, 1, size(x, 2))
    sampler_rng8 = Random.MersenneTwister(24680)
    sampled8 = metropolis_samples(
        uniform_net8,
        model8,
        mc_tcoef8,
        mc_Jcoef8;
        n_samples=128,
        n_chains=4,
        n_discard=8,
        sweeps_per_sample=1,
        rng=sampler_rng8,
    )

    @test length(sampled8.samples) == 128
    @test sampled8.proposed > 0
    @test 0.0 <= sampled8.acceptance <= 1.0
    @test sampled8.accepted <= sampled8.proposed
    @test all(length(state) == model8.M for state in sampled8.samples)
    @test all(count(==(Int8(1)), state) == model8.nup for state in sampled8.samples)
    @test all(count(==(Int8(-1)), state) == model8.ndown for state in sampled8.samples)
    @test all(count(==(Int8(0)), state) == model8.h for state in sampled8.samples)
    @test all(haskey(model8.hashmap, Tuple(state)) for state in sampled8.samples)

    for state in sampled8.samples[1:16]
        E_sparse = local_energy(uniform_net8, model8, state, mc_tcoef8, mc_Jcoef8; H=H8_mc)
        E_matrix_free = local_energy(uniform_net8, model8, state, mc_tcoef8, mc_Jcoef8; degrees=(1,))
        @test isapprox(E_matrix_free, E_sparse; atol=1.0e-8, rtol=1.0e-8)
    end

    exact_uniform_energy8 = Float64(variational(H8_mc, ones(Float32, model8.dim)))
    mc_uniform8 = mc_energy(
        uniform_net8,
        model8,
        mc_tcoef8,
        mc_Jcoef8;
        H=H8_mc,
        n_samples=1024,
        n_chains=8,
        n_discard=16,
        sweeps_per_sample=1,
        rng=Random.MersenneTwister(13579),
    )
    @test isfinite(mc_uniform8.energy)
    @test mc_uniform8.stderr >= 0
    @test isapprox(
        mc_uniform8.energy,
        exact_uniform_energy8;
        atol=max(0.35, 6 * mc_uniform8.stderr),
        rtol=0.0,
    )

    mc_params8 = copy(nn_params8)
    mc_params8["n_neurons"] = 16
    mc_params8["epochs"] = 1
    mc_params8["model_path"] = nothing
    mc_params8["load_from"] = nothing
    mc_params8["mc_samples"] = 16
    mc_params8["mc_eval_samples"] = 16
    mc_params8["mc_chains"] = 4
    mc_params8["mc_burn_in"] = 2
    mc_params8["mc_eval_burn_in"] = 2
    mc_params8["mc_learning_rate"] = 1.0e-3
    mc_params8["verbose"] = false
    mc_params8["return_wavefunction"] = false

    mc_train8 = trainNNMC(
        mc_params8,
        model8,
        mc_tcoef8,
        mc_Jcoef8,
        Ht_nn,
        HJ_nn;
        seed=97531,
        return_details=true,
    )
    @test isfinite(mc_train8.energy)
    @test length(mc_train8.energy_history) == 1
    @test mc_train8.psi === nothing
    @test all(0.0 .<= mc_train8.acceptance_history .<= 1.0)
end

Test Summary:                      | Pass  Total   Time
Monte Carlo sampling for 8-site tJ |   33     33  13.8s


Test.DefaultTestSet("Monte Carlo sampling for 8-site tJ", Any[], 33, false, false, true, 1.777951103388913e9, 1.777951117220681e9, false, "In[11]", Xoshiro(0x1fe117b804676271, 0xffad3c6eb0b8f26b, 0x4d1b74ebb6a6417f, 0xc620c1ebf4fff184, 0x0b5763125ad11bc9))

In [17]:
mc_nn_params8 = copy(nn_params8)
mc_nn_params8["model_path"] = "model8_tJ_nn_mc.jls"
mc_nn_params8["load_from"] = nothing
mc_nn_params8["mc_learning_rate"] = 1.0e-3
mc_nn_params8["mc_samples"] = 2048
mc_nn_params8["mc_eval_samples"] = 4096
mc_nn_params8["mc_chains"] = 128
mc_nn_params8["mc_burn_in"] = 64
mc_nn_params8["mc_eval_burn_in"] = 128
mc_nn_params8["mc_sweeps_per_sample"] = 1
mc_nn_params8["return_wavefunction"] = false
mc_nn_params8["verbose"] = true
mc_nn_params8["exploit_symmetry"] = false

training_tcoef8_mc = -t_value
training_Jcoef8_mc = J_value

# This is the actual Monte Carlo training call. It is intentionally left unrun here.
mc_train_nn8 = trainNNMC(
    mc_nn_params8,
    model8,
    training_tcoef8_mc,
    training_Jcoef8_mc,
    Ht_nn,
    HJ_nn;
    seed=12345,
    return_details=true,
)

psi0_mc_nn8 = mc_train_nn8.psi
E0_mc_nn8 = mc_train_nn8.energy
E0_mc_stderr8 = mc_train_nn8.stderr
exact_E0_8 = @isdefined(lowest_eigenvalues8) ? minimum(real.(lowest_eigenvalues8)) : missing
energy_error_mc8 = exact_E0_8 === missing ? missing : E0_mc_nn8 - exact_E0_8
overlap_mc8 = (@isdefined(lowest_eigenvectors8) && psi0_mc_nn8 !== nothing) ?
    maximum(abs.(lowest_eigenvectors8' * psi0_mc_nn8)) :
    missing
norm_psi0_mc8 = psi0_mc_nn8 === nothing ? missing : norm(psi0_mc_nn8)
mean_acceptance_mc8 = isempty(mc_train_nn8.acceptance_history) ?
    missing :
    sum(mc_train_nn8.acceptance_history) / length(mc_train_nn8.acceptance_history)

(
    E0_mc_nn8=E0_mc_nn8,
    E0_mc_stderr8=E0_mc_stderr8,
    exact_E0_8=exact_E0_8,
    energy_error_mc8=energy_error_mc8,
    overlap_mc8=overlap_mc8,
    norm_psi0_mc8=norm_psi0_mc8,
    mean_acceptance_mc8=mean_acceptance_mc8,
)

VMC neural network training device: gpu
VMC batched amplitude/local-energy path: true
epoch 5 completed, maximum remaining epochs: 495	MC energy: -3.444438341977458 +/- 0.040082857404774415	acceptance: 0.6938
epoch 10 completed, maximum remaining epochs: 490	MC energy: -3.501947801552888 +/- 0.03903895449035272	acceptance: 0.7501
epoch 15 completed, maximum remaining epochs: 485	MC energy: -3.5931816652728914 +/- 0.03706344335452005	acceptance: 0.7198
epoch 20 completed, maximum remaining epochs: 480	MC energy: -3.709946979582441 +/- 0.03726088476414424	acceptance: 0.6791
epoch 25 completed, maximum remaining epochs: 475	MC energy: -3.9151116524628486 +/- 0.03434127086251714	acceptance: 0.6227
epoch 30 completed, maximum remaining epochs: 470	MC energy: -4.121595377812002 +/- 0.028033679818040207	acceptance: 0.5651
epoch 35 completed, maximum remaining epochs: 465	MC energy: -4.233133357920076 +/- 0.025057598867782777	acceptance: 0.5342
epoch 40 completed, maximum remaining epochs: 460

(E0_mc_nn8 = -4.992929828746282, E0_mc_stderr8 = 0.006272047110289661, exact_E0_8 = -5.049420618984206, energy_error_mc8 = 0.05649079023792325, overlap_mc8 = missing, norm_psi0_mc8 = missing, mean_acceptance_mc8 = 0.5381729980468749)